# 6 Screening Comparison and Candidate Shortlisting

## 6.0 Purpose

This notebook compares the broad screening results for open and click prediction, assigns research roles to candidate models, and shortlists a small number of candidates for later tuning.

This stage does not tune models, use the final chronological test set, run SHAP, or select the final thesis model.

Historical feature leakage audit:
- current row outcome excluded via cumsum - current outcome
- mailing_id used as chronological proxy
- send_date incomplete
- ordering proxy previously checked and strongly aligned with available send_date
- residual risk acknowledged

## 6.1 Setup

In [1]:
# imports
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# project paths

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "Outputs"

print("Project root:")
print(PROJECT_ROOT)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nOutput directory exists:", OUTPUT_DIR.exists())

Project root:
/Users/khanhhien/Documents/Thesis

Output directory:
/Users/khanhhien/Documents/Thesis/Outputs

Output directory exists: True


In [3]:
# inspect available output files
print("Screening result files:")
for path in sorted(OUTPUT_DIR.glob("*screening_results*")):
    print("-", path.name)

print("\nBaseline result files:")
for path in sorted(OUTPUT_DIR.glob("*baseline_results*")):
    print("-", path.name)

print("\nCandidate spec files:")
for path in sorted(OUTPUT_DIR.glob("*candidate_specs*")):
    print("-", path.name)

Screening result files:
- 4_screening_results_open_v1.csv
- 5_screening_results_click_v1.csv

Baseline result files:
- 4_baseline_results_open_v1.csv
- 5_baseline_results_click_v1.csv

Candidate spec files:
- 4_candidate_specs_open_v1.csv
- 5_candidate_specs_click_v1.csv


## 6.2 Load screening outputs

In [4]:
# load screening outputs
open_results = pd.read_csv(
    OUTPUT_DIR / "4_screening_results_open_v1.csv"
)

open_baselines = pd.read_csv(
    OUTPUT_DIR / "4_baseline_results_open_v1.csv"
)

open_specs = pd.read_csv(
    OUTPUT_DIR / "4_candidate_specs_open_v1.csv"
)

click_results = pd.read_csv(
    OUTPUT_DIR / "5_screening_results_click_v1.csv"
)

click_baselines = pd.read_csv(
    OUTPUT_DIR / "5_baseline_results_click_v1.csv"
)

click_specs = pd.read_csv(
    OUTPUT_DIR / "5_candidate_specs_click_v1.csv"
)

print("OPEN RESULTS")
display(open_results.head())

print("\nOPEN BASELINES")
display(open_baselines)

print("\nOPEN SPECS")
display(open_specs.head())

print("\nCLICK RESULTS")
display(click_results.head())

print("\nCLICK BASELINES")
display(click_baselines)

print("\nCLICK SPECS")
display(click_specs.head())

OPEN RESULTS


,target,model_name,feature_set,candidate_key,notes,runtime_seconds,mean_ROC_AUC,std_ROC_AUC,mean_PR_AUC,std_PR_AUC,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,mean_brier,std_brier,mean_log_loss,std_log_loss
0,open,random_forest,C_expanded,random_forest__C_expanded,stable nonlinear expanded ensemble,73.294483,0.836392,0.026861,0.859215,0.037682,0.763761,0.028511,0.753004,0.031621,0.163052,0.017376,0.496900,0.043166
1,open,random_forest,A_core,random_forest__A_core,stable nonlinear ensemble,61.100862,0.830207,0.027462,0.844957,0.042158,0.764031,0.028647,0.751236,0.030757,0.165146,0.016516,0.504229,0.039897
2,open,decision_tree,C_expanded,decision_tree__C_expanded,simple nonlinear expanded model,17.451925,0.826182,0.036542,0.832031,0.054667,0.761653,0.026444,0.752640,0.030120,0.166246,0.019685,0.513710,0.053633
3,open,logistic_regression,C_expanded,logistic_regression__C_expanded,expanded logistic model,38.098379,0.821929,0.026887,0.836937,0.051186,0.753391,0.025303,0.742066,0.029898,0.170663,0.017311,0.519516,0.046379
4,open,decision_tree,A_core,decision_tree__A_core,simple nonlinear interpretable model,12.689436,0.819507,0.034983,0.816039,0.055258,0.760199,0.023182,0.750755,0.026124,0.166817,0.017365,0.517497,0.049015



OPEN BASELINES


,model_name,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss
0,dummy_majority,0.5,0.57075,0.57075,0.5,0.429250,15.471741
1,dummy_prior,0.5,0.57075,0.57075,0.5,0.245664,0.684472



OPEN SPECS


...

,target,model_name,feature_set,candidate_key,notes,runtime_seconds,mean_ROC_AUC,std_ROC_AUC,mean_PR_AUC,std_PR_AUC,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,mean_brier,std_brier,mean_log_loss,std_log_loss
0,click,random_forest,C_expanded,random_forest__C_expanded,stable nonlinear expanded ensemble,95.732967,0.927475,0.022739,0.508435,0.088197,0.999028,0.000318,0.649010,0.021786,0.000815,0.000220,0.009183,0.004519
1,click,random_forest_balanced,C_expanded,random_forest_balanced__C_expanded,class-weighted random forest,89.104418,0.919088,0.026233,0.447213,0.092001,0.998909,0.000362,0.590378,0.023875,0.000876,0.000256,0.010217,0.005324
2,click,logistic_regression,C_expanded,logistic_regression__C_expanded,expanded logistic model,38.351653,0.956329,0.016601,0.386815,0.102058,0.998809,0.000395,0.612044,0.023670,0.000947,0.000284,0.004403,0.001547
3,click,random_forest,A_core,random_forest__A_core,stable nonlinear ensemble,100.850984,0.916744,0.019652,0.360430,0.117603,0.998866,0.000323,0.597736,0.026678,0.000962,0.000256,0.010550,0.004464
4,click,logistic_regression_balanced,C_expanded,logistic_regression_balanced__C_expanded,class-weighted logistic,104.264547,0.942382,0.014207,0.353431,0.087197,0.950122,0.036043,0.908582,0.028609,0.044087,0.023586,0.175385,0.061835



CLICK BASELINES


,model_name,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss
0,dummy_majority,0.5,0.001252,0.998748,0.5,0.001252,0.045131
1,dummy_prior,0.5,0.001252,0.998748,0.5,0.001251,0.009661



CLICK SPECS


,model_name,feature_set,notes
0,logistic_regression,A_core,interpretable benchmark
1,logistic_regression,B_interactions,tests explicit interactions
2,logistic_regression,C_expanded,expanded logistic model
3,logistic_regression_balanced,C_expanded,class-weighted logistic
4,decision_tree,A_core,simple nonlinear interpretable model


In [5]:
# check columns
print("OPEN RESULTS")
print(open_results.columns.tolist())

print("\nCLICK RESULTS")
print(click_results.columns.tolist())

print("\nOPEN BASELINES")
print(open_baselines.columns.tolist())

print("\nCLICK BASELINES")
print(click_baselines.columns.tolist())

OPEN RESULTS
['target', 'model_name', 'feature_set', 'candidate_key', 'notes', 'runtime_seconds', 'mean_ROC_AUC', 'std_ROC_AUC', 'mean_PR_AUC', 'std_PR_AUC', 'mean_accuracy', 'std_accuracy', 'mean_balanced_accuracy', 'std_balanced_accuracy', 'mean_brier', 'std_brier', 'mean_log_loss', 'std_log_loss']

CLICK RESULTS
['target', 'model_name', 'feature_set', 'candidate_key', 'notes', 'runtime_seconds', 'mean_ROC_AUC', 'std_ROC_AUC', 'mean_PR_AUC', 'std_PR_AUC', 'mean_accuracy', 'std_accuracy', 'mean_balanced_accuracy', 'std_balanced_accuracy', 'mean_brier', 'std_brier', 'mean_log_loss', 'std_log_loss']

OPEN BASELINES
['model_name', 'ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'brier', 'log_loss']

CLICK BASELINES
['model_name', 'ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'brier', 'log_loss']


## 6.3 Build comparison tables

In [6]:
# open comparison table
open_comparison = open_results.copy()

open_comparison["primary_metric"] = "ROC_AUC"
open_comparison["primary_metric_mean"] = open_comparison["mean_ROC_AUC"]
open_comparison["primary_metric_std"] = open_comparison["std_ROC_AUC"]
open_comparison["primary_metric_conservative"] = (
    open_comparison["mean_ROC_AUC"] - open_comparison["std_ROC_AUC"]
)

open_comparison = open_comparison.sort_values(
    by="primary_metric_mean",
    ascending=False
).reset_index(drop=True)

open_cols = [
    "target",
    "model_name",
    "feature_set",
    "primary_metric",
    "primary_metric_mean",
    "primary_metric_std",
    "primary_metric_conservative",
    "mean_ROC_AUC",
    "mean_PR_AUC",
    "mean_brier",
    "mean_log_loss",
    "runtime_seconds",
    "notes"
]

display(open_comparison[open_cols])

,target,model_name,feature_set,primary_metric,primary_metric_mean,primary_metric_std,primary_metric_conservative,mean_ROC_AUC,mean_PR_AUC,mean_brier,mean_log_loss,runtime_seconds,notes
0,open,random_forest,C_expanded,ROC_AUC,0.836392,0.026861,0.809531,0.836392,0.859215,0.163052,0.496900,73.294483,stable nonlinear expanded ensemble
1,open,random_forest,A_core,ROC_AUC,0.830207,0.027462,0.802745,0.830207,0.844957,0.165146,0.504229,61.100862,stable nonlinear ensemble
2,open,decision_tree,C_expanded,ROC_AUC,0.826182,0.036542,0.789640,0.826182,0.832031,0.166246,0.513710,17.451925,simple nonlinear expanded model
3,open,logistic_regression,C_expanded,ROC_AUC,0.821929,0.026887,0.795042,0.821929,0.836937,0.170663,0.519516,38.098379,expanded logistic model
4,open,decision_tree,A_core,ROC_AUC,0.819507,0.034983,0.784524,0.819507,0.816039,0.166817,0.517497,12.689436,simple nonlinear interpretable model
5,open,logistic_regression,B_interactions,ROC_AUC,0.815858,0.029914,0.785944,0.815858,0.825610,0.172656,0.526053,16.015746,tests explicit interactions
6,open,logistic_regression,A_core,ROC_AUC,0.814210,0.031477,0.782733,0.814210,0.820981,0.172750,0.526848,20.392323,interpretable benchmark


In [7]:
# click comparison table
click_comparison = click_results.copy()

click_dummy_prior_pr_auc = click_baselines.loc[
    click_baselines["model_name"] == "dummy_prior",
    "PR_AUC"
].iloc[0]

click_comparison["primary_metric"] = "PR_AUC"
click_comparison["primary_metric_mean"] = click_comparison["mean_PR_AUC"]
click_comparison["primary_metric_std"] = click_comparison["std_PR_AUC"]
click_comparison["primary_metric_conservative"] = (
    click_comparison["mean_PR_AUC"] - click_comparison["std_PR_AUC"]
)

click_comparison["pr_auc_lift_over_dummy_prior"] = (
    click_comparison["mean_PR_AUC"] / click_dummy_prior_pr_auc
)

click_comparison = click_comparison.sort_values(
    by="primary_metric_mean",
    ascending=False
).reset_index(drop=True)

click_cols = [
    "target",
    "model_name",
    "feature_set",
    "primary_metric",
    "primary_metric_mean",
    "primary_metric_std",
    "primary_metric_conservative",
    "pr_auc_lift_over_dummy_prior",
    "mean_PR_AUC",
    "mean_ROC_AUC",
    "mean_brier",
    "mean_log_loss",
    "runtime_seconds",
    "notes"
]

display(click_comparison[click_cols])

,target,model_name,feature_set,primary_metric,primary_metric_mean,primary_metric_std,primary_metric_conservative,pr_auc_lift_over_dummy_prior,mean_PR_AUC,mean_ROC_AUC,mean_brier,mean_log_loss,runtime_seconds,notes
0,click,random_forest,C_expanded,PR_AUC,0.508435,0.088197,0.420239,406.056047,0.508435,0.927475,0.000815,0.009183,95.732967,stable nonlinear expanded ensemble
1,click,random_forest_balanced,C_expanded,PR_AUC,0.447213,0.092001,0.355213,357.161740,0.447213,0.919088,0.000876,0.010217,89.104418,class-weighted random forest
2,click,logistic_regression,C_expanded,PR_AUC,0.386815,0.102058,0.284757,308.925663,0.386815,0.956329,0.000947,0.004403,38.351653,expanded logistic model
3,click,random_forest,A_core,PR_AUC,0.360430,0.117603,0.242827,287.853277,0.360430,0.916744,0.000962,0.010550,100.850984,stable nonlinear ensemble
4,click,logistic_regression_balanced,C_expanded,PR_AUC,0.353431,0.087197,0.266234,282.263288,0.353431,0.942382,0.044087,0.175385,104.264547,class-weighted logistic
5,click,logistic_regression,A_core,PR_AUC,0.319062,0.117017,0.202045,254.815216,0.319062,0.950452,0.001022,0.004660,32.914503,interpretable benchmark
6,click,logistic_regression,B_interactions,PR_AUC,0.318563,0.126030,0.192533,254.416495,0.318563,0.951785,0.001018,0.004666,20.754242,tests explicit interactions
7,click,decision_tree,C_expanded,PR_AUC,0.193967,0.088585,0.105382,154.909150,0.193967,0.725973,0.001752,0.054831,45.523842,simple nonlinear expanded model
8,click,decision_tree,A_core,PR_AUC,0.161292,0.072962,0.088330,128.814264,0.161292,0.708614,0.001758,0.059328,31.025461,simple nonlinear interpretable model
9,click,decision_tree_balanced,C_expanded,PR_AUC,0.143465,0.057860,0.085606,114.576997,0.143465,0.665881,0.001599,0.050981,40.067780,class-weighted decision tree


## 6.4 Assign research roles

The purpose of the research role assignment is to clarify why each candidate model was included in the screening stage and what question it is intended to answer. Models are therefore evaluated not only based on predictive performance but also on their contribution to understanding feature engineering choices, model complexity, and interpretability-performance trade-offs.

Research objectives addressed by these candidates include:

* Whether expanded feature sets improve predictive performance compared with the core feature set.
* Whether manually specified interaction effects provide incremental predictive value.
* Whether nonlinear models outperform interpretable linear models.
* Whether the increase in predictive performance from more complex models justifies the corresponding loss of interpretability.

### 6.4.1 Open prediction

Logistic Regression Family
- Logistic A: Core interpretable benchmark using the thesis-aligned feature set
- Logistic B:Interaction-effect experiment evaluating whether manually specified interaction terms improve predictive performance
- Logistic C: Main interpretable model using the expanded feature set

Decision Tree Family
- Decision Tree A: Simple rule-based benchmark providing high interpretability
- Decision Tree C: Expanded nonlinear interpretable model evaluating whether additional features improve tree-based performance

Random Forest Family
- Random Forest A: Nonlinear ensemble benchmark using the core feature set
- Random Forest C: Main predictive ensemble model using the expanded feature set

### 6.4.2 Click prediction

Logistic Regression Family
- Logistic A: Interpretable benchmark model using the core feature set
- Logistic B: Interaction-effect experiment to evaluate whether explicit interaction terms improve performance
- Logistic C: Main interpretable model using the expanded feature set
- Logistic Balanced C: Class imbalance robustness experiment

Decision Tree Family
- Decision Tree A: Simple interpretable benchmark
- Decision Tree C: Expanded nonlinear interpretable model
- Decision Tree Balanced C: Class imbalance robustness experiment

Random Forest Family
- Random Forest A: Nonlinear benchmark ensemble
- Random Forest C: Main predictive ensemble model
- Random Forest Balanced C: Class imbalance robustness experiment

## 6.5 Compare within model families

### 6.5.1 Open prediction

Logistic Regression Family

Model: ROC-AUC
- Logistic A: 0.814
- Logistic B: 0.816
- Logistic C: 0.822

Observations:

* Explicit interaction features produced only a marginal improvement over the core model.
* The expanded feature set achieved the strongest performance within the logistic regression family.
* The performance gain from A to C suggests that the additional behavioural and campaign-related variables provide useful predictive information.
* Performance differences within the family are relatively modest compared with those observed for click prediction.

Decision:

* Keep Logistic C as the primary interpretable candidate.
* Keep Logistic A as the core benchmark model.
* Drop Logistic B from further modelling because the interaction features provided little additional predictive value.

⸻

Decision Tree Family

Model: ROC-AUC
- Decision Tree A: 0.820
- Decision Tree C: 0.826

Observations:

* The expanded feature set improved predictive performance.
* Decision trees achieved competitive ROC-AUC values while maintaining relatively high interpretability.
* The performance difference between A and C is modest but consistent.

Decision:

* Keep Decision Tree C as the strongest decision tree candidate.
* Keep Decision Tree A as the simple interpretable benchmark.

⸻

Random Forest Family

Model: ROC-AUC
- Random Forest A: 0.830
= Random Forest C: 0.836

Observations:

* The expanded feature set improved predictive performance.
* Random Forest C achieved the highest ROC-AUC among all screened open-prediction candidates.
* The performance gain over Logistic C is present but not substantial, suggesting that nonlinear relationships provide some additional predictive value.

Decision:

* Keep Random Forest C as the primary predictive candidate.
* Keep Random Forest A as the nonlinear benchmark model.

### 6.5.2 Click prediction

Logistic Regression Family

Model: PR-AUC
- Logistic A: 0.319
- Logistic B: 0.319
- Logistic C: 0.387
- Logistic Balanced C: 0.353

Observations:

* Explicit interaction features provided virtually no improvement over the core model.
* The expanded feature set produced a substantial increase in PR-AUC.
* Class weighting increased balanced accuracy but reduced PR-AUC.
* The expanded unweighted logistic regression achieved the strongest overall performance within the family.

Decision:

* Keep Logistic C as the primary interpretable candidate.
* Keep Logistic A as benchmark.
* Drop Logistic B from further modelling.
* Keep Logistic Balanced C as robustness check.

⸻

Decision Tree Family

Model: PR-AUC
- Decision Tree A: 0.161
- Decision Tree C: 0.194
- Decision Tree Balanced C: 0.143

Observations:

* Expanded features improved performance.
* Class weighting reduced performance.
* Decision trees remained substantially weaker than logistic regression and random forests.

Decision:

* Keep Decision Tree C.
* Keep Decision Tree A as interpretability benchmark.
* Drop Decision Tree Balanced C.

⸻

Random Forest Family

Model: PR-AUC
- Random Forest A: 0.360
- Random Forest C: 0.508
- Random Forest Balanced C: 0.447

Observations:

* Expanded features produced a major performance improvement.
* Class weighting reduced PR-AUC.
* Random Forest C achieved the strongest performance among all screened candidates.

Decision:

* Keep Random Forest C as primary predictive candidate.
* Keep Random Forest A as benchmark.
* Drop Random Forest Balanced C.

### 6.5.3 Secondary metric review

The primary metrics used for candidate selection were ROC-AUC for open prediction and PR-AUC for click prediction. Secondary metrics were reviewed to verify that the conclusions obtained from the primary metrics were not contradicted by calibration-related measures. Do secondary metrics tell a different story? Are the finalist actually robust?

- Open: PR-AUC, Brier, log loss
- Click: ROC-AUC, Brier, log loss

- Open finalists: Logistic C, Decision Tree C, Random Forest C
- Click finalists: Logistic C, Decision Tree C, Random Forest C

In [8]:
# open finalists
open_finalists = open_results[
    open_results["candidate_key"].isin([
        "logistic_regression__C_expanded",
        "decision_tree__C_expanded",
        "random_forest__C_expanded"
    ])
].copy()

open_secondary = open_finalists[
    [
        "candidate_key",
        "mean_ROC_AUC",
        "mean_PR_AUC",
        "mean_brier",
        "mean_log_loss",
        "runtime_seconds"
    ]
].sort_values(
    "mean_ROC_AUC",
    ascending=False
)

display(open_secondary)

,candidate_key,mean_ROC_AUC,mean_PR_AUC,mean_brier,mean_log_loss,runtime_seconds
0,random_forest__C_expanded,0.836392,0.859215,0.163052,0.496900,73.294483
2,decision_tree__C_expanded,0.826182,0.832031,0.166246,0.513710,17.451925
3,logistic_regression__C_expanded,0.821929,0.836937,0.170663,0.519516,38.098379


Open prediction

The comparison of the strongest candidate from each model family showed a consistent ranking across ROC-AUC, PR-AUC, Brier score, and log loss. Random Forest C achieved the strongest performance across all evaluation metrics, while Decision Tree C and Logistic C produced similar but slightly weaker results. No contradictions were observed between discrimination and calibration metrics.

In [9]:
# click finalists
click_finalists = click_results[
    click_results["candidate_key"].isin([
        "logistic_regression__C_expanded",
        "decision_tree__C_expanded",
        "random_forest__C_expanded"
    ])
].copy()

click_secondary = click_finalists[
    [
        "candidate_key",
        "mean_PR_AUC",
        "mean_ROC_AUC",
        "mean_brier",
        "mean_log_loss",
        "runtime_seconds"
    ]
].sort_values(
    "mean_PR_AUC",
    ascending=False
)

display(click_secondary)

,candidate_key,mean_PR_AUC,mean_ROC_AUC,mean_brier,mean_log_loss,runtime_seconds
0,random_forest__C_expanded,0.508435,0.927475,0.000815,0.009183,95.732967
2,logistic_regression__C_expanded,0.386815,0.956329,0.000947,0.004403,38.351653
7,decision_tree__C_expanded,0.193967,0.725973,0.001752,0.054831,45.523842


Click prediction

The comparison of the strongest candidate from each model family again produced largely consistent conclusions. Random Forest C achieved the highest PR-AUC and the best Brier score, while Logistic C achieved the highest ROC-AUC and lowest log loss. Given the extreme class imbalance of the click prediction task, PR-AUC remained the primary decision criterion. Therefore, the secondary metrics did not alter the conclusions obtained from the primary metric comparison.

Conclusion: the secondary metric review did not reveal any evidence that would justify changing the rankings established by the primary evaluation metrics. The candidate shortlist therefore proceeds unchanged to the final selection stage.

### 6.5.3 Tier 3 metrics review

Accuracy and balanced accuracy were reviewed as supplementary diagnostics.

For the open prediction task, class proportions were relatively balanced, making accuracy and balanced accuracy more interpretable than in the click prediction task. However, these metrics provide less information than ROC-AUC because they depend on a fixed classification threshold. The rankings established by ROC-AUC remained unchanged and no contradictory evidence was observed from the Tier 3 metrics.

For the click prediction task, accuracy is heavily influenced by the extreme class imbalance and therefore was not used for model selection. Balanced accuracy was useful for understanding the behaviour of class-weighted models but was not considered a primary decision criterion. The rankings established by PR-AUC remained unchanged.

## 6.6 Compare across model families

### 6.6.1 Open prediction

The strongest candidate from each model family was selected for cross-family comparison.

Model Family - Candidate: ROC-AUC
- Logistic Regression - Logistic C: 0.822
- Decision Tree	- Decision Tree C: 0.826
- Random Forest	- Random Forest C: 0.836

Observations:

* Decision Tree C achieved a modest improvement over Logistic C, suggesting the presence of nonlinear relationships in the data.
* Random Forest C achieved the strongest performance among all open-prediction candidates.
* The improvement from Logistic C to Random Forest C was relatively small in absolute terms.
* Logistic regression therefore remains competitive despite its substantially higher interpretability.

Interpretation:

The results suggest that open prediction benefits from nonlinear modelling approaches, but the advantage over logistic regression is not substantial. Consequently, both interpretability and predictive performance remain important considerations for the final model selection process.

### 6.6.2 Click prediction

The strongest candidate from each model family was selected for cross-family comparison.

Model Family - Candidate: PR-AUC
- Logistic Regression - Logistic C: 0.387
- Decision Tree - Decision Tree C: 0.194
- Random Forest	- Random Forest C: 0.508

Observations:

* Decision Tree C performed substantially worse than Logistic C and Random Forest C.
* Random Forest C achieved the strongest overall performance.
* The improvement from Logistic C to Random Forest C was substantial.
* The gain in predictive performance suggests that click prediction contains nonlinear relationships that are not fully captured by logistic regression.

Interpretation:

The results indicate that click prediction benefits considerably from nonlinear ensemble methods. Nevertheless, Logistic C remains an important interpretable benchmark and provides a useful reference point when evaluating the trade-off between predictive performance and model transparency.

## 6.7 Assign shortlist decisions

The purpose of the shortlisting stage is to identify a small number of candidate models that justify further investment through hyperparameter tuning and final evaluation. Candidate selection was based primarily on the chosen evaluation metric, while also considering interpretability, model complexity, computational efficiency, and the research objectives of the study.



### 6.7.1 Open predictor

Candidate - Decision - Justification

- Logistic Regression A - Keep as Benchmark Only - Core interpretable benchmark using the thesis-aligned feature set.
- Logistic Regression B - Thesis History Only - Interaction features provided negligible improvement over Logistic A.
- Logistic Regression C - Shortlist for Tuning - Strongest logistic model and primary interpretable candidate.

- Decision Tree A - Keep as Benchmark Only - Simple rule-based benchmark with high interpretability.
- Decision Tree C - Shortlist for Tuning - Strongest decision tree candidate, provides nonlinear modelling while remaining interpretable and computationally efficient.

- Random Forest A - Keep as Benchmark Only - Nonlinear ensemble benchmark used to assess the value of feature expansion.
- Random Forest C - Shortlist for Tuning - Best overall predictive performance among screened open-prediction models.

Open Prediction Tuning Candidates

* Logistic Regression C
* Decision Tree C
* Random Forest C

### 6.7.2 Click prediction

Candidate - Decision - Justification
- Logistic Regression A - Keep as Benchmark Only - Core interpretable benchmark using the thesis-aligned feature set.
- Logistic Regression B - Thesis History Only	- Interaction features provided negligible improvement over Logistic A.
- Logistic Regression C - Shortlist for Tuning - Strongest logistic model and primary interpretable candidate.
- Logistic Regression Balanced C - Keep as Robustness Check - Useful for evaluating the effect of class weighting under extreme class imbalance.
- Decision Tree A - Keep as Benchmark Only - Simple rule-based benchmark.
Decision Tree C	- Thesis History Only - Improved over Tree A but remained substantially weaker than shortlisted candidates.
Decision Tree Balanced C - Thesis History Only - Class weighting did not improve performance.
Random Forest A	- Keep as Benchmark Only - Nonlinear ensemble benchmark used to assess the value of feature expansion.
Random Forest C	- Shortlist for Tuning - Strongest overall click-prediction model.
Random Forest Balanced C - Thesis History Only - Class weighting reduced primary-metric performance.

Click Prediction Tuning Candidates

* Logistic Regression C
* Random Forest C

### 6.7. 3 Tuning strategy

Only shortlisted candidates will proceed to hyperparameter tuning.

Benchmark models will remain fixed at their screening-stage specifications and will be retained for comparison purposes.

Robustness-check models may be re-evaluated if required for methodological validation but will not undergo full hyperparameter tuning.

Models assigned to thesis history remain documented in the screening results but will not receive additional modelling effort.

## 6.8 Save outputs

In [10]:
# save comparison tables
open_comparison.to_csv(
    OUTPUT_DIR / "6_open_comparison_v1.csv",
    index=False
)

click_comparison.to_csv(
    OUTPUT_DIR / "6_click_comparison_v1.csv",
    index=False
)

print("Comparison tables saved.")

Comparison tables saved.


In [11]:
# open shortlist
open_shortlist = pd.DataFrame({
    "candidate_key": [
        "logistic_regression__A_core",
        "logistic_regression__B_interactions",
        "logistic_regression__C_expanded",
        "decision_tree__A_core",
        "decision_tree__C_expanded",
        "random_forest__A_core",
        "random_forest__C_expanded"
    ],
    "decision": [
        "BENCHMARK",
        "THESIS_HISTORY",
        "SHORTLIST",
        "BENCHMARK",
        "SHORTLIST",
        "BENCHMARK",
        "SHORTLIST"
    ]
})

display(open_shortlist)

,candidate_key,decision
0,logistic_regression__A_core,BENCHMARK
1,logistic_regression__B_interactions,THESIS_HISTORY
2,logistic_regression__C_expanded,SHORTLIST
3,decision_tree__A_core,BENCHMARK
4,decision_tree__C_expanded,SHORTLIST
5,random_forest__A_core,BENCHMARK
6,random_forest__C_expanded,SHORTLIST


In [12]:
# click shortlist
click_shortlist = pd.DataFrame({
    "candidate_key": [
        "logistic_regression__A_core",
        "logistic_regression__B_interactions",
        "logistic_regression__C_expanded",
        "logistic_regression_balanced__C_expanded",
        "decision_tree__A_core",
        "decision_tree__C_expanded",
        "decision_tree_balanced__C_expanded",
        "random_forest__A_core",
        "random_forest__C_expanded",
        "random_forest_balanced__C_expanded"
    ],
    "decision": [
        "BENCHMARK",
        "THESIS_HISTORY",
        "SHORTLIST",
        "ROBUSTNESS_CHECK",
        "BENCHMARK",
        "THESIS_HISTORY",
        "THESIS_HISTORY",
        "BENCHMARK",
        "SHORTLIST",
        "THESIS_HISTORY"
    ]
})

display(click_shortlist)

,candidate_key,decision
0,logistic_regression__A_core,BENCHMARK
1,logistic_regression__B_interactions,THESIS_HISTORY
2,logistic_regression__C_expanded,SHORTLIST
3,logistic_regression_balanced__C_expanded,ROBUSTNESS_CHECK
4,decision_tree__A_core,BENCHMARK
5,decision_tree__C_expanded,THESIS_HISTORY
6,decision_tree_balanced__C_expanded,THESIS_HISTORY
7,random_forest__A_core,BENCHMARK
8,random_forest__C_expanded,SHORTLIST
9,random_forest_balanced__C_expanded,THESIS_HISTORY


In [13]:
# save shortlist tables
open_shortlist.to_csv(
    OUTPUT_DIR / "6_open_shortlist_v1.csv",
    index=False
)

click_shortlist.to_csv(
    OUTPUT_DIR / "6_click_shortlist_v1.csv",
    index=False
)

print("Shortlist tables saved.")

Shortlist tables saved.


In [14]:
# stage 6 completion summary
print("=" * 60)
print("STAGE 6 COMPLETE")
print("=" * 60)

print("\nOPEN TUNING CANDIDATES")
print("- logistic_regression__C_expanded")
print("- decision_tree__C_expanded")
print("- random_forest__C_expanded")

print("\nCLICK TUNING CANDIDATES")
print("- logistic_regression__C_expanded")
print("- random_forest__C_expanded")

print("\nReady for Stage 7: Hyperparameter Tuning")

STAGE 6 COMPLETE

OPEN TUNING CANDIDATES
- logistic_regression__C_expanded
- decision_tree__C_expanded
- random_forest__C_expanded

CLICK TUNING CANDIDATES
- logistic_regression__C_expanded
- random_forest__C_expanded

Ready for Stage 7: Hyperparameter Tuning
